## Polytope Climate-DT Time Series example notebook

This notebook shows how to use earthkit-data and earthkit-plots to pull destination-earth data from LUMI and plot it using earthkit-plots.

Before running the notebook you need to set up your credentials. See the main readme of this repository for different ways to do this or use the following cells to authenticate.

You will need to generate your credentials using the desp-authentication.py script.

This can be run as follows:

In [ ]:
%%capture cap
%run ../../desp-authentication.py

This will generate a token that can then be used by earthkit and polytope.

In [ ]:
output_1 = cap.stdout.split('}\n')
access_token = output_1[-1][0:-1]

# Requirements
To run this notebook install the following:
* pip install earthkit-data
* pip install earthkit-plots
* pip install earthkit-regrid  (Optional for spectral variables)
* pip install cf-units         (Optional for unit conversion in maps)

If you do not have eccodes installed please install eccodes using conda as it is a dependency of earthkit, or install earthkit via conda

* conda install eccodes -c conda-forge
* conda install earthkit-data -c conda-forge

In [ ]:
import earthkit.data
import earthkit.plots as ekp

In [ ]:
# Defaults to making a live data request. Set to false to use the cached GRIB file instead.
import os

LIVE_REQUEST = os.getenv("LIVE_REQUEST", "true").lower() == "true"
LIVE_REQUEST

In [ ]:
LOCATION = ((38, -9.5))

In [ ]:
request = {
    "activity": "projections",
    "class": "d1",
    "dataset": "climate-dt",
    "experiment": "ssp3-7.0",
    "generation": "2",
    "levtype": "sfc",
    "date": "20400101/to/20410101",
    "model": "ifs-nemo",
    "expver": "0001",
    "param": "167",
    "realization": "1",
    "resolution": "standard",
    "stream": "clte",
    "type": "fc",
    "time": "0000", # "time": "0000/to/2300/by/0100" for hourly
    "feature": {
        "type" : "timeseries",
        "points": [[LOCATION[0], LOCATION[1]]],
        "time_axis": "date",
    }
}

In [ ]:
data_file = "../data/climate-dt-earthkit-fe-timeseries.covjson"
if LIVE_REQUEST:
    data = earthkit.data.from_source("polytope", "destination-earth", request, address="polytope.mn5.apps.dte.destination-earth.eu", stream=False)
    with open(data_file, "w") as _f:
        data.to_target("file", _f)
else:
    data = earthkit.data.from_source("file", data_file) 

In [ ]:
def location_to_string(location):
    """
    Converts latitude and longitude to a string representation with degrees
    and N/S/E/W.
    """
    (lat, lon) = location
    lat_dir = "N" if lat >= 0 else "S"
    lon_dir = "E" if lon >= 0 else "W"
    return f"{abs(lat):.2f}°{lat_dir}, {abs(lon):.2f}°{lon_dir}"

In [ ]:
ds = data.to_xarray()
ds

<xarray.Dataset> Size: 6kB
Dimensions:    (latitude: 1, longitude: 1, levelist: 1, number: 1, datetime: 1,
                t: 367)
Coordinates:
  * latitude   (latitude) float64 8B 37.92
  * longitude  (longitude) float64 8B 350.5
  * levelist   (levelist) int64 8B 0
  * number     (number) int64 8B 0
  * datetime   (datetime) <U20 80B '2041-01-01 00:00:00Z'
  * t          (t) datetime64[ns] 3kB 2040-01-01 2040-01-02 ... 2041-01-01
Data variables:
    2t         (latitude, longitude, levelist, number, datetime, t) float64 3kB ...
Attributes: (12/14)
    activity:       projections
    class:          d1
    dataset:        climate-dt
    experiment:     ssp3-7.0
    expver:         0001
    generation:     2
    ...             ...
    realization:    1
    resolution:     standard
    stream:         clte
    type:           fc
    number:         0
    Forecast date:  2041-01-01 00:00:00Z

In [ ]:
chart = ekp.timeseries.line(
    ds.squeeze(),
    units="celsius",
    color='grey',
    title=f"ECMWF ensemble meteogram at {location_to_string(LOCATION)}",
    xticks={
        "frequency": "M",
        "format": "%b",
        "period": True,
    },
    linewidth=1,
)
chart.show()